# Visual Verification on t5_HD_720p

Regenerates `t5_scrambled.png` and `t5_restored.png` for the configured grid (Python server encrypts, Node.js `igrt_client.js` decrypts) and verifies bit-exact recovery. Outputs are written to `docs/images/` and `figures/`.

In [ ]:
import json
import random
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src" / "cryptography"))
import grt_server

# ----- Configuration: edit this block to change images, grids, or run counts -----
SCENARIOS = [
    {"name": "SD 480p",   "file": "docs/images/t1_SD_480p.png",   "M": 15, "N": 14},
    {"name": "FHD 1080p", "file": "docs/images/t2_FHD_1080p.png", "M": 30, "N": 20},
    {"name": "HD 720p",   "file": "docs/images/t5_HD_720p.png",   "M": 18, "N": 20},
]
WARMUP, ITERS, SEED = 5, 30, 2026
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def make_keys(M, N):
    """Reproducible valid GRT keys (a, b, c, d, rm, rn)."""
    random.seed(SEED)
    np.random.seed(SEED)
    keys = tuple(int(k) for k in grt_server.generate_valid_grt_keys(M, N))
    ok, msg = grt_server.validate_grt_parameters(M, N, *keys)
    assert ok, msg
    return keys


def load_scenario(sc):
    """Load image, pad (edge replication) to a multiple of the grid, return padded image and original size."""
    img = Image.open(ROOT / sc["file"]).convert("RGB")
    padded, orig_size = grt_server.pad_image_to_grid(img, sc["M"], sc["N"])
    return padded, orig_size


def stats(values):
    arr = np.array(values)
    return {"mean": float(arr.mean()), "std": float(arr.std()), "median": float(np.median(arr))}

In [ ]:
IGRT_JS = ROOT / "src" / "cryptography" / "igrt_client.js"

JS_RUNNER = """
const fs = require('fs');
const igrt = require(process.argv[2]);
const [inPath, outPath, w, h, M, N, a, b, c, d, rm, rn, iters, warm] =
    [process.argv[3], process.argv[4], ...process.argv.slice(5).map(Number)];
const src32 = new Uint32Array(new Uint8Array(fs.readFileSync(inPath)).buffer);
const dst32 = new Uint32Array(src32.length);
for (let i = 0; i < warm; i++) igrt.descrambleBuffer32IGRT(src32, dst32, w, h, M, N, a, b, c, d, rm, rn);
const times = [];
for (let i = 0; i < iters; i++) {
    const t0 = process.hrtime.bigint();
    igrt.descrambleBuffer32IGRT(src32, dst32, w, h, M, N, a, b, c, d, rm, rn);
    times.push(Number(process.hrtime.bigint() - t0) / 1e6);
}
if (outPath !== '-') fs.writeFileSync(outPath, Buffer.from(dst32.buffer));
console.log(JSON.stringify(times));
"""


def run_igrt_client(scrambled, M, N, keys, iters=ITERS, warm=WARMUP, want_image=False):
    """Run the real igrt_client.js (descrambleBuffer32IGRT) in Node.js. Returns (timings_ms, restored_image|None)."""
    w, h = scrambled.size
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "in.bin").write_bytes(scrambled.convert("RGBA").tobytes())
        (tmp / "run.js").write_text(JS_RUNNER, encoding="utf-8")
        out_path = str(tmp / "out.bin") if want_image else "-"
        cmd = ["node", str(tmp / "run.js"), str(IGRT_JS).replace("\\", "/"), str(tmp / "in.bin"), out_path,
               str(w), str(h), str(M), str(N), *map(str, keys), str(iters), str(warm)]
        res = subprocess.run(cmd, capture_output=True, text=True, check=True)
        timings = json.loads(res.stdout.strip().splitlines()[-1])
        restored = None
        if want_image:
            restored = Image.frombytes("RGBA", (w, h), (tmp / "out.bin").read_bytes()).convert("RGB")
    return timings, restored

In [ ]:
sc = next(s for s in SCENARIOS if "t5" in s["file"])
M, N = sc["M"], sc["N"]
img, orig_size = load_scenario(sc)
keys = make_keys(M, N)

scrambled = grt_server.scramble_image_grt(img, M, N, *keys)
_, restored = run_igrt_client(scrambled, M, N, keys, iters=1, warm=0, want_image=True)
restored = restored.crop((0, 0, *orig_size))

orig_arr = np.array(img.crop((0, 0, *orig_size))).astype(int)
diff = np.abs(orig_arr - np.array(restored).astype(int))
mse = float((diff ** 2).mean())
print(f"Image {orig_size[0]}x{orig_size[1]}, grid {M}x{N}, tile {img.width // M}x{img.height // N}, keys {keys}")
print(f"Mismatched pixels: {int(np.count_nonzero(diff.sum(axis=-1)))}, max delta: {int(diff.max())}, MSE: {mse}")
assert mse == 0.0

for folder in ("docs/images", "figures"):
    target = ROOT / folder
    target.mkdir(exist_ok=True)
    scrambled.save(target / "t5_scrambled.png")
    restored.save(target / "t5_restored.png")
print("Saved t5_scrambled.png and t5_restored.png to docs/images/ and figures/")